## **Training**

In [ ]:
import os
from pathlib import Path
import time
import numpy as np
import pandas as pd
import sys
ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'kkthn').is_dir() and (path / 'notebooks').is_dir()
)
SRC = ROOT / 'kkthn' / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from kkthn import KKTHardNet


In [ ]:
# ----------------------------
# Config
# ----------------------------
DATA_PATH = "dataset/ED_Col_Data.csv"
WORK_PATH = "dataset/ED_Col_Data_2000.csv"

PARAMETERS = ["x1", "x2", "x3"]
VARIABLES = ["y1", "y2", "y3", "y4", "y5", "y6", "y7", "y8", "y9"]

TRAIN = {
    "epochs": 1200,
    "batch_size": 40,
    "learning_rate": 1e-3,
    "train_frac": 0.8,          # 80% train, 20% validation
    "hidden_size": 64,
    "hidden_layers": 2,
    "seed": 42,
    "dtype": "float64",
    "print_every": 100,
    "newton_step_length": 0.5,
    "newton_tol": 1e-6,
    "newton_reg_factor": 1e-3,
    "max_newton_iter": 30,
    "max_backtrack_iter": 10,
    "eta": 1e-4,
    "epoch_mlp": 100,
    "cons_alpha": 10
}


In [ ]:

# ----------------------------
# Load data, select 2000 points
# ----------------------------
df = pd.read_csv(DATA_PATH)

required_cols = PARAMETERS + VARIABLES
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

df_2000 = (
    df[required_cols]
    .dropna()
    .sample(n=2000, random_state=TRAIN["seed"])
    .reset_index(drop=True)
)

os.makedirs("dataset", exist_ok=True)
df_2000.to_csv(WORK_PATH, index=False)

parameters_csv = "dataset/parameters_2000.csv"
variables_csv = "dataset/variables_2000.csv"

df_2000[PARAMETERS].to_csv(parameters_csv, index=False)
df_2000[VARIABLES].to_csv(variables_csv, index=False)



In [ ]:
# ----------------------------
# Build KKT-HardNet model
# ----------------------------
model = KKTHardNet(name="ED_Column", train=TRAIN)

x = model.add_parameter(PARAMETERS)
y = model.add_variable(VARIABLES)

model.constraints.add(
    x.x1 + x.x2 - y.y1 - y.y2 == 0,
    x.x1 * 0.697616946 - y.y1 * y.y3 - y.y2 * y.y6 == 0,
    x.x1 * 0.302383054 - y.y1 * y.y4 - y.y2 * y.y7 == 0,
    y.y3 + y.y4 + y.y5 - 1 == 0,
    y.y6 + y.y7 + y.y8 - 1 == 0,
    x.x3 * y.y1 - y.y9 == 0,
)

model.dataset(
    parameters=parameters_csv,
    variables=variables_csv,
)


In [ ]:
# ----------------------------
# Train surrogate model
# ----------------------------
result = model.model()

print("Training finished.")



In [ ]:
# ----------------------------
# Check generated output folder
# ----------------------------
run_dirs = [
    d for d in os.listdir(".")
    if os.path.isdir(d) and d.startswith("ED_Column")
]

In [ ]:
model.summary()

In [ ]:
model.plot_history(bg="white")

## **Inference**

In [ ]:
import os
from pathlib import Path
import time
import numpy as np
import pandas as pd
import sys
ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'kkthn').is_dir() and (path / 'notebooks').is_dir()
)
SRC = ROOT / 'kkthn' / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from kkthn import KKTHardNet


In [ ]:
# ----------------------------
# Get latest run directory
# ----------------------------
DATA_PATH = "dataset/ED_Col_Data.csv"
PARAMETERS = ["x1", "x2", "x3"]
VARIABLES = ["y1", "y2", "y3", "y4", "y5", "y6", "y7", "y8", "y9"]

run_dirs = [
    d for d in os.listdir(".")
    if os.path.isdir(d) and d.startswith("ED_Column_")
]

latest_run_dir = max(run_dirs, key=os.path.getmtime)

metadata_path = os.path.join(latest_run_dir, "metadata.json")

df = pd.read_csv(DATA_PATH)


In [ ]:
# ----------------------------
# Load model
# ----------------------------
loaded_model = KKTHardNet()
loaded_model.load(metadata_path)

# ----------------------------
# Prepare inputs
# ----------------------------
single_x = df[PARAMETERS].iloc[0].tolist()
batch_x = df[PARAMETERS].iloc[:40].values.tolist()


In [ ]:
# ----------------------------
# Warm-up (JAX compilation)
# ----------------------------
loaded_model.predict(single_x, projection_backend="jax")
loaded_model.predict(batch_x, projection_backend="jax")

# ----------------------------
# Single inference benchmark
# ----------------------------
single_times = []

for _ in range(100):
    start = time.perf_counter()
    loaded_model.predict(single_x)
    end = time.perf_counter()
    single_times.append(end - start)

single_times = np.array(single_times)
print(f"Single inference: {single_times.mean():.6e} ± {single_times.std():.6e} seconds")


In [ ]:
# ----------------------------
# Batch inference benchmark
# ----------------------------
batch_times = []

for _ in range(100):
    start = time.perf_counter()
    loaded_model.predict(batch_x)
    end = time.perf_counter()
    batch_times.append(end - start)

batch_times = np.array(batch_times)
print(f"Batch inference (40 samples): {batch_times.mean():.6e} ± {batch_times.std():.6e} seconds")
